# AI Fraud & Bank-Application Risk Intelligence

**Explainable fraud detection with human-readable features, temporal validation, operational thresholds and fairness checks.**

## Business problem
A retail bank cannot manually review every new account application. This project ranks applications by fraud risk and answers:

1. Which applications should investigators review first?
2. Which understandable applicant, device and behavioural signals drive risk?
3. How does the model perform on a later month rather than a random split?
4. How much fraud can be captured at limited review capacity?
5. Do false-positive rates differ across age groups?

## Dataset
This notebook uses the **Feedzai Bank Account Fraud (BAF) Dataset Suite — Base variant (NeurIPS 2022)**.

- 1,000,000 bank-account-opening applications
- 32 columns including the binary target `fraud_bool`
- 8 monthly periods (`month = 0..7`)
- ~1.1% fraud
- readable features rather than PCA labels such as V1–V28
- privacy-preserving **synthetic** data generated from an anonymised real-world bank-account fraud dataset

Upstream project: `feedzai/bank-account-fraud`. Kaggle is the upstream dataset distribution; the notebook first tries OpenML and then an unofficial Hugging Face raw mirror for reproducible execution.

> The project deliberately describes BAF as synthetic-from-real rather than claiming that the 1M rows are raw customer records.

## Modelling stack
- leakage-safe chronological split: months 0–5 train, 6 validation, 7 test
- one-hot encoding for categorical signals
- class-balanced SGD logistic baseline
- XGBoost rare-event classifier
- PR-AUC and ROC-AUC model comparison
- operating point chosen on validation data at <=5% false-positive rate
- capture@K / review-capacity analysis
- SHAP global and local explanations
- age-group false-positive-rate audit
- investigator-ready ranked queue


In [ ]:
# Colab / GitHub Actions safe setup
import sys, subprocess, importlib.util

for pkg in ["xgboost", "shap"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Environment ready.")


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score, recall_score,
    f1_score, confusion_matrix, precision_recall_curve, roc_curve,
    classification_report
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Libraries imported.")


## 1. Load BAF with interpretable fraud features

The loader prefers an OpenML copy when available and otherwise downloads the Base CSV from a public reproducibility mirror. The official Feedzai/Kaggle source remains the dataset authority.


In [ ]:
HF_BASE_URL = "https://huggingface.co/datasets/jAEhEEkIM/operationbench-baf-raw/resolve/main/Base.csv"

try:
    baf = fetch_openml(name="BAF_base", version=1, as_frame=True)
    df = baf.frame.copy()
    if "fraud_bool" not in df.columns and baf.target is not None:
        df["fraud_bool"] = pd.Series(baf.target, index=df.index)
    data_route = "OpenML: BAF_base"
except Exception as exc:
    print("OpenML route unavailable; using public BAF raw mirror.")
    print("OpenML message:", str(exc)[:180])
    df = pd.read_csv(HF_BASE_URL)
    data_route = "Hugging Face raw mirror of upstream BAF Base.csv"

df["fraud_bool"] = pd.to_numeric(df["fraud_bool"], errors="raise").astype(int)
df["month"] = pd.to_numeric(df["month"], errors="raise").astype(int)

required = {
    "fraud_bool", "month", "income", "name_email_similarity", "customer_age",
    "credit_risk_score", "foreign_request", "source", "device_os",
    "employment_status", "housing_status", "proposed_credit_limit"
}
missing_required = required.difference(df.columns)
if missing_required:
    raise RuntimeError(f"Required BAF fields missing: {sorted(missing_required)}")

print("Data route:", data_route)
print(f"Shape: {df.shape}")
print(f"Fraud cases: {int(df['fraud_bool'].sum()):,}")
print(f"Fraud rate: {df['fraud_bool'].mean():.4%}")
print("Months:", sorted(df["month"].unique().tolist()))
display(df.head())


## 2. Human-readable feature dictionary


In [ ]:
feature_dictionary = pd.DataFrame([
    ("income", "Applicant income quantile (0–1 scale)"),
    ("name_email_similarity", "Similarity between applicant name and email"),
    ("prev_address_months_count", "Months at previous registered address"),
    ("current_address_months_count", "Months at current registered address"),
    ("customer_age", "Applicant age bucket"),
    ("days_since_request", "Days since request/application activity"),
    ("intended_balcon_amount", "Intended initial balance/transfer amount"),
    ("payment_type", "Application payment type"),
    ("velocity_6h", "Recent application velocity over 6 hours"),
    ("velocity_24h", "Recent application velocity over 24 hours"),
    ("velocity_4w", "Recent application velocity over 4 weeks"),
    ("employment_status", "Applicant employment category"),
    ("credit_risk_score", "Credit-risk score"),
    ("email_is_free", "Whether email provider is a free service"),
    ("housing_status", "Applicant housing category"),
    ("phone_home_valid", "Home phone validity flag"),
    ("phone_mobile_valid", "Mobile phone validity flag"),
    ("bank_months_count", "Length of banking relationship"),
    ("has_other_cards", "Whether applicant has other cards"),
    ("proposed_credit_limit", "Proposed credit limit"),
    ("foreign_request", "Whether request originated from another country"),
    ("source", "Application source: browser or app"),
    ("session_length_in_minutes", "Online application session duration"),
    ("device_os", "Applicant device operating system"),
    ("keep_alive_session", "Whether keep-alive session option was used"),
    ("device_distinct_emails_8w", "Distinct emails seen on device over 8 weeks"),
    ("month", "Application month; used only for temporal splitting"),
    ("fraud_bool", "Target: 1 fraud, 0 legitimate"),
], columns=["feature", "business_meaning"])

display(feature_dictionary)
print("No anonymous V1–V28 columns are used.")


## 3. Data-quality and class-imbalance audit


In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_null": df.isna().sum(),
    "unique": df.nunique(dropna=False)
})
print(f"Null values: {int(df.isna().sum().sum()):,}")
print(f"Exact duplicate rows: {int(df.duplicated().sum()):,}")
print(f"Rows: {len(df):,}")
display(quality.head(32))

counts = df["fraud_bool"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.bar(["Legitimate", "Fraud"], counts.values)
ax.set_title("Bank-account application class imbalance")
ax.set_ylabel("Applications")
for i, value in enumerate(counts.values):
    ax.text(i, value, f"{value:,}\n({value/len(df):.2%})", ha="center", va="bottom")
plt.tight_layout()
plt.show()

naive_accuracy = 1 - df["fraud_bool"].mean()
print(f"An always-legitimate classifier would be {naive_accuracy:.2%} accurate — therefore accuracy is not the primary metric.")


## 4. Exploratory fraud signals that an interviewer can understand


In [ ]:
monthly = df.groupby("month")["fraud_bool"].agg(["count", "sum", "mean"]).rename(
    columns={"sum":"frauds", "mean":"fraud_rate"}
)
display(monthly)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(monthly.index, monthly["fraud_rate"] * 100, marker="o")
ax.set_title("Fraud rate changes over time")
ax.set_xlabel("Month")
ax.set_ylabel("Fraud rate (%)")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Name/email mismatch is directly interpretable.
sim_bins = pd.cut(df["name_email_similarity"], bins=np.linspace(0, 1, 11), include_lowest=True)
sim_risk = df.groupby(sim_bins, observed=False)["fraud_bool"].agg(["count","mean"])
display(sim_risk)

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.bar(range(len(sim_risk)), sim_risk["mean"] * 100)
ax.set_xticks(range(len(sim_risk)), [str(x) for x in sim_risk.index], rotation=45, ha="right")
ax.set_title("Fraud rate by name–email similarity")
ax.set_xlabel("Name–email similarity band")
ax.set_ylabel("Fraud rate (%)")
plt.tight_layout()
plt.show()


In [ ]:
def category_fraud_table(column):
    out = df.groupby(column)["fraud_bool"].agg(["count","sum","mean"]).rename(
        columns={"sum":"frauds", "mean":"fraud_rate"}
    ).sort_values("fraud_rate", ascending=False)
    return out

for col in ["device_os", "source", "employment_status", "housing_status", "foreign_request"]:
    print(f"\nFraud profile — {col}")
    display(category_fraud_table(col).style.format({"fraud_rate":"{:.2%}"}))


## 5. Leakage-safe chronological split

Random splitting would mix later applications into training. We instead mimic deployment:

- **Train:** months 0–5
- **Validation:** month 6
- **Test:** month 7

The `month` field itself is not used as a model feature.


In [ ]:
train_df = df[df["month"] <= 5].copy()
val_df = df[df["month"] == 6].copy()
test_df = df[df["month"] == 7].copy()

target = "fraud_bool"
categorical_cols = ["payment_type", "employment_status", "housing_status", "source", "device_os"]
exclude = [target, "month"]
feature_cols = [c for c in df.columns if c not in exclude]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

X_train, y_train = train_df[feature_cols], train_df[target]
X_val, y_val = val_df[feature_cols], val_df[target]
X_test, y_test = test_df[feature_cols], test_df[target]

split_summary = pd.DataFrame({
    "rows":[len(train_df), len(val_df), len(test_df)],
    "frauds":[int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
    "fraud_rate":[y_train.mean(), y_val.mean(), y_test.mean()]
}, index=["train_m0_5","validation_m6","test_m7"])

display(split_summary.style.format({"fraud_rate":"{:.3%}"}))
print("Numeric features:", len(numeric_cols))
print("Categorical features:", categorical_cols)


## 6. Preprocess readable numeric + categorical fields


In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(with_mean=False), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
], remainder="drop", sparse_threshold=1.0)

X_train_enc = preprocessor.fit_transform(X_train)
X_val_enc = preprocessor.transform(X_val)
X_test_enc = preprocessor.transform(X_test)

feature_names = np.array([
    name.replace("num__", "").replace("cat__", "")
    for name in preprocessor.get_feature_names_out()
])

print("Encoded train shape:", X_train_enc.shape)
print("Readable transformed features:", len(feature_names))
print("Examples:", feature_names[:20].tolist())


## 7. Train a linear baseline and XGBoost


In [ ]:
# Fast class-balanced probabilistic linear baseline.
sgd = SGDClassifier(
    loss="log_loss",
    class_weight="balanced",
    alpha=1e-4,
    max_iter=1200,
    tol=1e-3,
    random_state=RANDOM_STATE,
)
sgd.fit(X_train_enc, y_train)
val_prob_sgd = sgd.predict_proba(X_val_enc)[:, 1]

neg, pos = np.bincount(y_train.astype(int))
scale_pos_weight = neg / pos

xgb = XGBClassifier(
    n_estimators=250,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    reg_lambda=2.0,
    reg_alpha=0.05,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=2,
)
xgb.fit(X_train_enc, y_train)
val_prob_xgb = xgb.predict_proba(X_val_enc)[:, 1]

print(f"scale_pos_weight = {scale_pos_weight:.1f}")
print("Models trained.")


## 8. Validation model comparison

**PR-AUC** is primary because fraud is rare; ROC-AUC is retained as a complementary discrimination metric.


In [ ]:
def probability_metrics(name, y_true, prob):
    return {
        "model":name,
        "PR_AUC":average_precision_score(y_true, prob),
        "ROC_AUC":roc_auc_score(y_true, prob)
    }

validation_results = pd.DataFrame([
    probability_metrics("Balanced SGD logistic baseline", y_val, val_prob_sgd),
    probability_metrics("XGBoost", y_val, val_prob_xgb)
]).set_index("model").sort_values("PR_AUC", ascending=False)

display(validation_results.style.format("{:.4f}"))

probability_map = {
    "Balanced SGD logistic baseline": val_prob_sgd,
    "XGBoost": val_prob_xgb
}
best_model_name = validation_results.index[0]
val_prob_best = probability_map[best_model_name]
print("Selected on validation PR-AUC:", best_model_name)

fig, ax = plt.subplots(figsize=(8.5, 5.3))
for name, prob in probability_map.items():
    p, r, _ = precision_recall_curve(y_val, prob)
    ax.plot(r, p, label=f"{name} (AP={average_precision_score(y_val, prob):.3f})")
ax.set_title("Validation precision–recall curves")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()


## 9. Freeze an operational threshold on validation data

The BAF literature commonly frames performance around a **5% false-positive-rate ceiling**. Here the validation threshold is selected to maximise recall while staying at or below 5% FPR. The threshold is then frozen before the test month is touched.


In [ ]:
fpr, tpr, thresholds = roc_curve(y_val, val_prob_best)
eligible = np.where(fpr <= 0.05)[0]
best_idx = eligible[np.argmax(tpr[eligible])]
operating_threshold = float(thresholds[best_idx])

val_pred = (val_prob_best >= operating_threshold).astype(int)
val_cm = confusion_matrix(y_val, val_pred)
tn, fp, fn, tp = val_cm.ravel()

print(f"Frozen operating threshold: {operating_threshold:.6f}")
print(f"Validation FPR: {fp/(fp+tn):.2%}")
print(f"Validation recall: {tp/(tp+fn):.2%}")
print(f"Validation precision: {tp/(tp+fp):.2%}")
print(f"Validation alerts: {int(val_pred.sum()):,} / {len(val_pred):,}")

fig, ax = plt.subplots(figsize=(5.4, 4.5))
im = ax.imshow(val_cm)
ax.set_title("Validation confusion matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_xticks([0,1], ["Legitimate","Fraud"])
ax.set_yticks([0,1], ["Legitimate","Fraud"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{val_cm[i,j]:,}", ha="center", va="center")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


## 10. One-time test-month evaluation


In [ ]:
test_prob_sgd = sgd.predict_proba(X_test_enc)[:, 1]
test_prob_xgb = xgb.predict_proba(X_test_enc)[:, 1]
test_prob_map = {
    "Balanced SGD logistic baseline": test_prob_sgd,
    "XGBoost": test_prob_xgb
}
test_prob = test_prob_map[best_model_name]
test_pred = (test_prob >= operating_threshold).astype(int)

test_pr_auc = average_precision_score(y_test, test_prob)
test_roc_auc = roc_auc_score(y_test, test_prob)
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_f1 = f1_score(y_test, test_pred, zero_division=0)
test_cm = confusion_matrix(y_test, test_pred)
tn, fp, fn, tp = test_cm.ravel()
test_fpr = fp / (fp + tn)

test_metrics = pd.DataFrame([{
    "selected_model":best_model_name,
    "PR_AUC":test_pr_auc,
    "ROC_AUC":test_roc_auc,
    "precision":test_precision,
    "recall":test_recall,
    "F1":test_f1,
    "false_positive_rate":test_fpr,
    "alerts":int(test_pred.sum())
}]).set_index("selected_model")

display(test_metrics.style.format({
    "PR_AUC":"{:.4f}", "ROC_AUC":"{:.4f}", "precision":"{:.2%}",
    "recall":"{:.2%}", "F1":"{:.4f}", "false_positive_rate":"{:.2%}"
}))

print(classification_report(
    y_test, test_pred,
    target_names=["Legitimate","Fraud"],
    zero_division=0
))

fig, ax = plt.subplots(figsize=(5.4, 4.5))
im = ax.imshow(test_cm)
ax.set_title("Month-7 test confusion matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_xticks([0,1], ["Legitimate","Fraud"])
ax.set_yticks([0,1], ["Legitimate","Fraud"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{test_cm[i,j]:,}", ha="center", va="center")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


## 11. Investigator capacity: how much fraud is found in the highest-risk applications?


In [ ]:
def capture_by_review_rate(y_true, score, review_rates=(0.005, 0.01, 0.02, 0.05)):
    ranked = pd.DataFrame({"actual":np.asarray(y_true), "score":np.asarray(score)})
    ranked = ranked.sort_values("score", ascending=False).reset_index(drop=True)
    total_fraud = ranked["actual"].sum()
    rows = []
    for rate in review_rates:
        k = max(1, int(np.ceil(len(ranked) * rate)))
        found = int(ranked.head(k)["actual"].sum())
        rows.append({
            "review_rate":rate,
            "applications_reviewed":k,
            "frauds_found":found,
            "precision_in_queue":found/k,
            "fraud_capture_rate":found/total_fraud if total_fraud else np.nan
        })
    return pd.DataFrame(rows)

capacity = capture_by_review_rate(y_test, test_prob)
display(capacity.style.format({
    "review_rate":"{:.1%}",
    "precision_in_queue":"{:.2%}",
    "fraud_capture_rate":"{:.2%}"
}))

fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.plot(capacity["review_rate"]*100, capacity["fraud_capture_rate"]*100, marker="o")
ax.set_title("Fraud capture vs manual-review capacity")
ax.set_xlabel("Applications reviewed (%)")
ax.set_ylabel("Fraud captured (%)")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 12. Explain what the model learned — readable feature importance + SHAP


In [ ]:
importance = pd.Series(xgb.feature_importances_, index=feature_names).sort_values(ascending=False)
top_importance = importance.head(20)

fig, ax = plt.subplots(figsize=(9, 6.5))
top_importance.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Top XGBoost fraud-risk features")
ax.set_xlabel("Model importance")
plt.tight_layout()
plt.show()
display(top_importance.to_frame("importance"))


In [ ]:
import shap

rng = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(X_test_enc.shape[0], size=min(800, X_test_enc.shape[0]), replace=False)
X_shap = X_test_enc[sample_idx]
X_shap_dense = X_shap.toarray() if hasattr(X_shap, "toarray") else np.asarray(X_shap)

explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_shap_dense)

shap.summary_plot(
    shap_values,
    X_shap_dense,
    feature_names=feature_names,
    max_display=18,
    show=False
)
plt.title("SHAP: which readable signals push fraud risk up or down?")
plt.tight_layout()
plt.show()


## 13. Local reason codes for a high-risk application


In [ ]:
top_pos = int(np.argmax(test_prob))
top_encoded = X_test_enc[top_pos]
top_dense = top_encoded.toarray() if hasattr(top_encoded, "toarray") else np.asarray(top_encoded)
local_shap = np.asarray(explainer.shap_values(top_dense)).reshape(-1)
local_values = np.asarray(top_dense).reshape(-1)

local_reasons = pd.DataFrame({
    "feature":feature_names,
    "encoded_value":local_values,
    "SHAP_contribution":local_shap,
    "absolute_impact":np.abs(local_shap)
}).sort_values("absolute_impact", ascending=False).head(12)

print(f"Highest-risk test application score: {test_prob[top_pos]:.4f}")
display(local_reasons[["feature","encoded_value","SHAP_contribution"]])

original_row = test_df.iloc[[top_pos]].copy()
show_fields = [
    "income", "name_email_similarity", "customer_age", "credit_risk_score",
    "current_address_months_count", "employment_status", "housing_status",
    "foreign_request", "source", "device_os", "proposed_credit_limit",
    "session_length_in_minutes", "fraud_bool"
]
display(original_row[[c for c in show_fields if c in original_row.columns]])


## 14. Fairness / error-distribution audit

This is an **audit**, not a claim that age causes fraud. We compare false-positive and true-positive rates across age buckets to identify whether model errors warrant deeper investigation.


In [ ]:
audit = test_df[["customer_age", "income", "fraud_bool"]].copy()
audit["prediction"] = test_pred
audit["risk_score"] = test_prob

def group_error_metrics(frame, group_col):
    rows = []
    for group, g in frame.groupby(group_col):
        y = g["fraud_bool"].to_numpy()
        p = g["prediction"].to_numpy()
        tn, fp, fn, tp = confusion_matrix(y, p, labels=[0,1]).ravel()
        rows.append({
            group_col:group,
            "n":len(g),
            "fraud_rate":y.mean(),
            "alert_rate":p.mean(),
            "false_positive_rate":fp/(fp+tn) if (fp+tn) else np.nan,
            "true_positive_rate":tp/(tp+fn) if (tp+fn) else np.nan
        })
    return pd.DataFrame(rows).sort_values(group_col)

age_audit = group_error_metrics(audit, "customer_age")
display(age_audit.style.format({
    "fraud_rate":"{:.2%}",
    "alert_rate":"{:.2%}",
    "false_positive_rate":"{:.2%}",
    "true_positive_rate":"{:.2%}"
}))

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.bar(age_audit["customer_age"].astype(str), age_audit["false_positive_rate"]*100)
ax.set_title("False-positive rate by applicant age bucket")
ax.set_xlabel("Customer age bucket")
ax.set_ylabel("False-positive rate (%)")
plt.tight_layout()
plt.show()

print("Interpretation rule: investigate disparities; do not treat this descriptive audit as proof of discrimination or causality.")


## 15. Investigator-ready queue with understandable columns


In [ ]:
queue = test_df.copy()
queue["risk_score"] = test_prob
queue["alert"] = test_pred
queue["risk_percentile"] = queue["risk_score"].rank(pct=True)
queue["risk_tier"] = pd.cut(
    queue["risk_percentile"],
    bins=[0, 0.95, 0.98, 0.995, 1.0],
    labels=["Low","Medium","High","Critical"],
    include_lowest=True
)
queue = queue.sort_values("risk_score", ascending=False)

queue_fields = [
    "risk_score", "risk_tier", "alert", "fraud_bool",
    "income", "name_email_similarity", "customer_age", "credit_risk_score",
    "current_address_months_count", "employment_status", "housing_status",
    "foreign_request", "source", "device_os", "proposed_credit_limit",
    "session_length_in_minutes"
]
queue_fields = [c for c in queue_fields if c in queue.columns]

print("Top 25 applications for fraud-team review:")
display(queue[queue_fields].head(25))

queue[queue_fields].to_csv("bank_fraud_investigation_queue.csv", index=False)
capacity.to_csv("bank_fraud_review_capacity.csv", index=False)
age_audit.to_csv("bank_fraud_age_audit.csv", index=False)
validation_results.to_csv("bank_fraud_validation_models.csv")
print("Operational artefacts exported.")


## 16. Executive summary


In [ ]:
cap_2 = capacity.iloc[(capacity["review_rate"] - 0.02).abs().argsort()[:1]].iloc[0]

print(f"""
EXECUTIVE BANK-APPLICATION FRAUD SUMMARY
----------------------------------------
Dataset rows: {len(df):,}
Readable raw model features: {len(feature_cols)}
Selected model: {best_model_name}
Frozen validation threshold: {operating_threshold:.6f}

MONTH-7 TEST PERFORMANCE
PR-AUC: {test_pr_auc:.4f}
ROC-AUC: {test_roc_auc:.4f}
Precision: {test_precision:.2%}
Recall: {test_recall:.2%}
F1: {test_f1:.4f}
False-positive rate: {test_fpr:.2%}
Alerts generated: {int(test_pred.sum()):,}

AT ~2% MANUAL REVIEW CAPACITY
Applications reviewed: {int(cap_2['applications_reviewed']):,}
Frauds found: {int(cap_2['frauds_found']):,}
Queue precision: {cap_2['precision_in_queue']:.2%}
Fraud captured: {cap_2['fraud_capture_rate']:.2%}

WHY THIS IS STRONGER THAN V1–V28
The model now works with interpretable applicant, credit, behavioural and
device fields, uses a chronological holdout, freezes its operating threshold
before final testing, quantifies review capacity, generates SHAP explanations,
and audits error rates across age groups.
""")


## Production roadmap

- probability calibration using a later validation window;
- cost-sensitive thresholding with bank-specific investigation and fraud-loss estimates;
- drift monitoring by month for fraud rate, score distribution and high-impact features;
- feature freshness/latency checks before scoring;
- champion/challenger deployment;
- human-investigator feedback loop;
- additional fairness audits and policy review before protected characteristics influence decisions;
- model registry, reproducible data versions and signed evaluation reports;
- API serving with reason codes rather than opaque scores alone.

### Interview defence
**“I replaced anonymous PCA variables with a realistic bank-account fraud benchmark containing interpretable application, credit, behavioural and device signals. I used a chronological train/validation/test split, compared a linear baseline with XGBoost, froze the operating threshold on validation data, measured capture at fixed investigation capacity, explained scores with SHAP, and audited false-positive rates across age groups.”**
